# Across-Plot Analysis of Visualization Comment Categories

**Data source:** `ace_classifications/` — LLM-classified ACE sentences from reader comments on 26 data visualizations.

We classify each ACE statement into categories that capture *observations* extracted from the chart, *prior knowledge* retrieved by the viewer, and subsequent *inferences* drawn to form a coherent mental model. Meta/paratext and uncategorizable statements are filtered out.

**Three high-level groups:**
- **Visual Observation** — Chart Structure & Text (VO1), Data Point Extraction (VO2), Cross-point Pattern Recognition (VO3)
- **Prior Knowledge & Evaluative** — Background, Personal/Episodic, Evaluative: Prescriptive, Evaluative: Reactive
- **Inference & Curiosity** — Explanatory, Predictive/Hypothetical, Curiosity

**Research questions:**
1. **Beyond-plot consistency** — How consistently do people speak to information that goes beyond the plot (prior knowledge, personal context, hypothetical reasoning)?
2. **Beyond-plot content** — When they go beyond the plot, is it prior knowledge or hypothetical reasoning? Which sub-categories dominate?
3. **Carving visual observations** — How do the three VO sub-types distribute across plots?
4. **Within-category divergence** — Are there diverging/contradictory statements in the same category on the same plot?

In [14]:
from __future__ import annotations

import json
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
alt.data_transformers.enable("vegafusion")

# ── Load all ace_classifications into a single DataFrame ────────────────
CLASSIFICATIONS_DIR = Path("..") / "ace_classifications"

rows = []
for p in sorted(CLASSIFICATIONS_DIR.glob("*.json")):
    with p.open() as f:
        rows.extend(json.load(f))

df_raw = pd.DataFrame(rows)
df_raw["article_id"] = df_raw["article_id"].astype(str)
df_raw["comment_id"] = df_raw["comment_id"].astype(int)

# Normalise stray legacy tags into their canonical form
_TAG_CLEANUP = {
    "L3: Trend and pattern analysis": "Visual Observation: Cross-point Pattern Recognition",
    "L1: Elemental and encoded properties": "Visual Observation: Chart Structure & Text",
    "L2: Statistical concepts and relations": "Visual Observation: Data Point Extraction",
    "VO1: Chart Structure, Layout & Text": "Visual Observation: Chart Structure & Text",
    "VO2: Data Point Reading": "Visual Observation: Data Point Extraction",
    "VO3: Comparisons, Trends & Patterns": "Visual Observation: Cross-point Pattern Recognition",
    "Background knowledge": "Prior Knowledge: Background",
    "Personal/episodic retrieval": "Prior Knowledge: Personal / Episodic",
    "Evaluative / affective judgment": "Evaluative: Reactive",
    "Explanatory inference": "Inference: Explanatory",
    "Predictive / counterfactual inference": "Inference: Predictive / Hypothetical",
    "Information need / curiosity": "Curiosity",
    "Meta /Paratext": "Meta / Paratext",
    "Meta / paratext": "Meta / Paratext",
}
df_raw["comment_tag"] = df_raw["comment_tag"].replace(_TAG_CLEANUP)

# Filter out Meta / Paratext and Uncategorizable
EXCLUDED = {"Meta / Paratext", "Uncategorizable"}
n_before = len(df_raw)
df = df_raw[~df_raw["comment_tag"].isin(EXCLUDED)].copy()
print(f"Filtered {n_before - len(df):,} Meta/Uncategorizable → {len(df):,} sentences retained")

# Short labels for plotting
SHORT = {
    "Visual Observation: Chart Structure & Text": "VO1",
    "Visual Observation: Data Point Extraction": "VO2",
    "Visual Observation: Cross-point Pattern Recognition": "VO3",
    "Prior Knowledge: Background": "Background",
    "Prior Knowledge: Personal / Episodic": "Personal",
    "Evaluative: Prescriptive": "Prescriptive",
    "Evaluative: Reactive": "Reactive",
    "Inference: Explanatory": "Explanatory",
    "Inference: Predictive / Hypothetical": "Predictive",
    "Curiosity": "Curiosity",
}
df["tag_short"] = df["comment_tag"].map(SHORT).fillna(df["comment_tag"])

# High-level groups matching the paper taxonomy
VISUAL_OBS_TAGS = {"VO1", "VO2", "VO3"}
PRIOR_KNOWLEDGE_TAGS = {"Background", "Personal", "Prescriptive", "Reactive"}
HYPOTHETICAL_TAGS = {"Explanatory", "Predictive", "Curiosity"}

def _group(tag_short):
    if tag_short in VISUAL_OBS_TAGS:
        return "Visual Observation"
    if tag_short in PRIOR_KNOWLEDGE_TAGS:
        return "Prior Knowledge & Evaluative"
    if tag_short in HYPOTHETICAL_TAGS:
        return "Inference & Curiosity"
    return "Other"

df["group"] = df["tag_short"].map(_group)

print(f"Loaded {len(df):,} sentences across {df['article_id'].nunique()} articles")
print()
print(df["group"].value_counts().to_string())
print()
df["comment_tag"].value_counts()

Filtered 79,056 Meta/Uncategorizable → 434,155 sentences retained
Loaded 434,155 sentences across 188 articles

group
Visual Observation              175986
Inference & Curiosity           134431
Prior Knowledge & Evaluative    123738



comment_tag
Visual Observation: Cross-point Pattern Recognition    108323
Curiosity                                               71608
Prior Knowledge: Background                             52876
Prior Knowledge: Personal / Episodic                    48955
Visual Observation: Chart Structure & Text              37448
Inference: Predictive / Hypothetical                    33797
Visual Observation: Data Point Extraction               30215
Inference: Explanatory                                  29026
Evaluative: Reactive                                    13281
Evaluative: Prescriptive                                 8626
Name: count, dtype: int64

## RQ1 — Beyond-plot consistency

*How consistently do people speak to information that goes beyond the plot?*

We define **"beyond-plot"** as any statement *not* classified as a Visual Observation. This includes two high-level groups:
- **Prior Knowledge & Evaluative** — Background, Personal/Episodic, Evaluative: Prescriptive, Evaluative: Reactive
- **Inference & Curiosity** — Explanatory, Predictive/Hypothetical, Curiosity

Below we compute the **proportion of beyond-plot sentences per article** and per commenter, then visualise variation across all 26 plots.

In [15]:
# ── Per-article group proportions ───────────────────────────────────────
GROUP_ORDER = ["Visual Observation", "Prior Knowledge & Evaluative", "Inference & Curiosity"]
GROUP_COLORS = ["#4c78a8", "#f58518", "#e45756"]

group_counts = (
    df.groupby(["article_id", "group"])
    .size()
    .unstack(fill_value=0)
)
group_props = group_counts.div(group_counts.sum(axis=1), axis=0)

summary = group_props[GROUP_ORDER].describe().T
print("── Group proportions across articles (summary stats) ──")
display(summary.round(3))

# Beyond-plot = everything that is not Visual Observation
group_props["Beyond-plot"] = 1 - group_props.get("Visual Observation", 0)
article_order = group_props["Beyond-plot"].sort_values(ascending=False).index.tolist()

props_long = group_props[GROUP_ORDER].reset_index().melt(
    id_vars="article_id", var_name="group", value_name="proportion"
)

alt.Chart(props_long, title="Sentence group proportions by article").mark_bar().encode(
    x=alt.X("article_id:N", sort=article_order, title="Article"),
    y=alt.Y("proportion:Q", stack="normalize", title="Proportion of sentences"),
    color=alt.Color(
        "group:N",
        scale=alt.Scale(domain=GROUP_ORDER, range=GROUP_COLORS),
        title="Group",
    ),
    tooltip=["article_id", "group", alt.Tooltip("proportion:Q", format=".1%")],
).properties(width=700, height=350)

── Group proportions across articles (summary stats) ──


,count,mean,std,min,25%,50%,75%,max
group,,,,,,,,
Visual Observation,188.0,0.401,0.081,0.121,0.352,0.395,0.440,0.653
Prior Knowledge & Evaluative,188.0,0.286,0.084,0.056,0.248,0.301,0.337,0.788
Inference & Curiosity,188.0,0.312,0.058,0.091,0.272,0.304,0.344,0.540


alt.Chart(...)

In [42]:
# ── Per-article top-level category proportions (excluding Meta and Uncategorizable) ─────────────

# Define fine-grained subcategories with explicit bottom-to-top stacking order:
# Visual Observation (blue; bottom), Prior Knowledge (red), Evaluative (green), Inference (yellow), Curiosity (brown; top)

FINEGRAIN_CATS = [
    "Visual Observation: Chart Structure & Text",         # light blue (bottom)
    "Visual Observation: Data Point Extraction",          # medium blue
    "Visual Observation: Cross-point Pattern Recognition",# darkest blue
    "Prior Knowledge: Background",                       # light red
    "Prior Knowledge: Personal / Episodic",              # dark red
    "Evaluative: Prescriptive",                          # light green
    "Evaluative: Reactive",                              # dark green
    "Inference: Explanatory",                            # light yellow
    "Inference: Predictive / Hypothetical",              # dark yellow
    "Curiosity",                                         # light brown (top)
]

FINEGRAIN_COLORS = [
    "#a9c5f2",  # Visual Observation: Chart Structure & Text (light blue)
    "#4c78a8",  # Visual Observation: Data Point Extraction   (medium blue)
    "#25476a",  # Visual Observation: Cross-point Pattern Recognition (darkest blue)
    "#f6b0b0",  # Prior Knowledge: Background (light red)
    "#b51230",  # Prior Knowledge: Personal / Episodic (dark red)
    "#8fe596",  # Evaluative: Prescriptive (light green)
    "#34803a",  # Evaluative: Reactive (dark green)
    "#ffe993",  # Inference: Explanatory (light yellow)
    "#e1b800",  # Inference: Predictive / Hypothetical (dark yellow)
    "#cbb293",  # Curiosity (light brown)
]

SUBCAT_TO_GROUP = {
    "Visual Observation: Chart Structure & Text": "Visual Observation",
    "Visual Observation: Data Point Extraction": "Visual Observation",
    "Visual Observation: Cross-point Pattern Recognition": "Visual Observation",
    "Prior Knowledge: Background": "Prior Knowledge",
    "Prior Knowledge: Personal / Episodic": "Prior Knowledge",
    "Evaluative: Prescriptive": "Evaluative Response",
    "Evaluative: Reactive": "Evaluative Response",
    "Inference: Explanatory": "Inference",
    "Inference: Predictive / Hypothetical": "Inference",
    "Curiosity": "Curiosity",
}

VALID_SUBCATS = FINEGRAIN_CATS  # Use explicit ordered list above

# Filter for only valid subcategories in stacked bar order
df_fine = df[df["comment_tag"].isin(VALID_SUBCATS)].copy()
df_fine["subcat"] = df_fine["comment_tag"]
df_fine["group"] = df_fine["subcat"].map(SUBCAT_TO_GROUP)

# Per-article, per-fine-grained-subcategory counts
fine_counts = (
    df_fine.groupby(["article_id", "subcat"])
    .size()
    .unstack(fill_value=0)
)

fine_props = fine_counts.div(fine_counts.sum(axis=1), axis=0)

summary = fine_props[FINEGRAIN_CATS].describe().T
print("── Subcategory proportions across articles (summary stats) ──")
display(summary.round(3))

# Used for consistent article ordering: sum all beyond-Visual Observation categories
vo_subcats = [
    "Visual Observation: Chart Structure & Text",
    "Visual Observation: Data Point Extraction",
    "Visual Observation: Cross-point Pattern Recognition",
]
fine_props["Beyond-plot"] = 1 - fine_props[vo_subcats].sum(axis=1)
article_order = fine_props["Beyond-plot"].sort_values(ascending=False).index.tolist()

# For plotting: proportions for tooltips, etc.
subcat_counts = (
    df_fine.groupby(["article_id", "group", "subcat"])
    .size()
    .reset_index(name="n")
)
subcat_counts["group_total"] = subcat_counts.groupby(["article_id", "group"])["n"].transform("sum")
subcat_counts["subcat_prop"] = subcat_counts["n"] / subcat_counts["group_total"]
subcat_counts["subcat_total"] = subcat_counts.groupby(["article_id"])["n"].transform("sum")
subcat_counts["top_prop"] = subcat_counts["n"] / subcat_counts["subcat_total"]

# Altair plot: stacked bars by subcategory (all visually distinct), with requested bottom-->top order & color
alt.Chart(
    subcat_counts,
    title=(
        "Sentence subcategory proportions by article\n"
        "(Stacked order: "
        "Visual Obs. (shades of blue, bottom), "
        "Prior Knowledge (red), Evaluative (green), "
        "Inference (yellow), Curiosity (brown, top))"
    )
).mark_bar().encode(
    x=alt.X("article_id:N", sort=article_order, title="Article"),
    y=alt.Y("top_prop:Q", stack="normalize", title="Proportion of sentences"),
    color=alt.Color(
        "subcat:N",
        sort=FINEGRAIN_CATS,
        scale=alt.Scale(domain=FINEGRAIN_CATS, range=FINEGRAIN_COLORS),
        title="Subcategory",
        legend=alt.Legend(
            orient="bottom",
            title="Subcategory"
        ),
    ),
    order=alt.Order('subcat:N', sort='descending'),  # bottom-to-top stacking order
    tooltip=[
        "article_id",
        "group",
        "subcat",
        alt.Tooltip("top_prop:Q", format=".1%", title="Article prop (of all)"),
        alt.Tooltip("subcat_prop:Q", format=".1%", title="Subcat prop (of group)"),
        "n"
    ],
).properties(width=800, height=350)

── Subcategory proportions across articles (summary stats) ──


,count,mean,std,min,25%,50%,75%,max
subcat,,,,,,,,
Visual Observation: Chart Structure & Text,188.0,0.089,0.049,0.000,0.056,0.079,0.111,0.368
Visual Observation: Data Point Extraction,188.0,0.067,0.034,0.000,0.042,0.061,0.089,0.204
Visual Observation: Cross-point Pattern Recognition,188.0,0.245,0.067,0.000,0.202,0.244,0.282,0.459
Prior Knowledge: Background,188.0,0.127,0.040,0.037,0.099,0.122,0.149,0.269
Prior Knowledge: Personal / Episodic,188.0,0.103,0.064,0.000,0.034,0.115,0.150,0.238
Evaluative: Prescriptive,188.0,0.022,0.026,0.000,0.009,0.017,0.027,0.303
Evaluative: Reactive,188.0,0.034,0.025,0.006,0.022,0.030,0.042,0.303
Inference: Explanatory,188.0,0.072,0.028,0.000,0.054,0.068,0.085,0.170
Inference: Predictive / Hypothetical,188.0,0.078,0.047,0.000,0.048,0.067,0.091,0.367


alt.Chart(...)

In [ ]:
# ── Beyond-plot proportion: distribution across articles ────────────────
bp_props = group_props["Beyond-plot"].reset_index()
bp_props.columns = ["article_id", "beyond_plot_pct"]

print(f"Beyond-plot proportion across {len(bp_props)} articles (excl. meta & uncategorizable):")
print(f"  Mean:   {bp_props['beyond_plot_pct'].mean():.1%}")
print(f"  Median: {bp_props['beyond_plot_pct'].median():.1%}")
print(f"  Std:    {bp_props['beyond_plot_pct'].std():.1%}")
print(f"  Range:  [{bp_props['beyond_plot_pct'].min():.1%}, {bp_props['beyond_plot_pct'].max():.1%}]")

# Per-group means
for g in GROUP_ORDER:
    col = group_props.get(g, pd.Series(dtype=float))
    print(f"  {g}: mean={col.mean():.1%}, std={col.std():.1%}")

strip = alt.Chart(bp_props, title="Beyond-plot proportion per article").mark_circle(size=80).encode(
    x=alt.X("beyond_plot_pct:Q", title="Proportion beyond-plot", axis=alt.Axis(format="%")),
    y=alt.value(0),
    tooltip=["article_id:N", alt.Tooltip("beyond_plot_pct:Q", format=".1%")],
)

rule = alt.Chart(bp_props).mark_rule(color="red", strokeDash=[4, 4]).encode(
    x="mean(beyond_plot_pct):Q"
)

(strip + rule).properties(width=600, height=60)

In [16]:
# ── Per-commenter beyond-plot rate (distribution within each article) ───
commenter_group = (
    df.groupby(["article_id", "comment_id", "group"])
    .size()
    .unstack(fill_value=0)
)
commenter_total = commenter_group.sum(axis=1)

# Beyond-plot = Prior Knowledge & Evaluative + Inference & Curiosity
commenter_bp_n = (
    commenter_group.get("Prior Knowledge & Evaluative", 0)
    + commenter_group.get("Inference & Curiosity", 0)
)
commenter_bp = (commenter_bp_n / commenter_total).reset_index()
commenter_bp.columns = ["article_id", "comment_id", "bp_frac"]

has_bp = commenter_bp.groupby("article_id")["bp_frac"].agg(
    n_commenters="count",
    pct_with_bp=lambda s: (s > 0).mean(),
    median_bp="median",
    mean_bp="mean",
)
print("── Per-article: how many commenters go beyond the plot? ──")
display(has_bp.sort_values("pct_with_bp", ascending=False).round(3))

alt.Chart(
    commenter_bp, title="Per-commenter beyond-plot fraction (by article)"
).mark_boxplot(extent="min-max").encode(
    x=alt.X("article_id:N", sort=article_order, title="Article"),
    y=alt.Y("bp_frac:Q", title="Beyond-plot fraction", axis=alt.Axis(format="%")),
    color=alt.Color("article_id:N", legend=None),
).properties(width=700, height=300)

── Per-article: how many commenters go beyond the plot? ──


,n_commenters,pct_with_bp,median_bp,mean_bp
article_id,,,,
181,153,1.000,0.556,0.579
154,255,1.000,0.714,0.690
94,258,1.000,0.700,0.685
170,250,1.000,0.636,0.632
7,23,1.000,0.750,0.755
...,...,...,...,...
43,300,0.797,0.400,0.424
12,110,0.782,0.423,0.463
2,458,0.755,0.308,0.338


alt.Chart(...)

## RQ2 — What do people comment on when they go beyond the plot?

Beyond-plot statements fall into two high-level groups:
- **Prior Knowledge & Evaluative**: Background, Personal/Episodic, Evaluative: Prescriptive, Evaluative: Reactive
- **Inference & Curiosity**: Explanatory, Predictive/Hypothetical, Curiosity

Below we break down the seven sub-categories to see which dominate and how they vary across articles.

In [17]:
# ── Beyond-plot sub-category breakdown ──────────────────────────────────
bp_df = df[df["group"] != "Visual Observation"].copy()

# Overall distribution
bp_overall = bp_df["tag_short"].value_counts()
print("── Overall beyond-plot sub-category counts ──")
display(bp_overall.to_frame("count").assign(
    group=bp_overall.index.map(lambda t: "Prior Knowledge & Evaluative" if t in PRIOR_KNOWLEDGE_TAGS else "Inference & Curiosity"),
    pct=lambda d: (d["count"] / d["count"].sum()).map("{:.1%}".format),
))

# Per-article breakdown (proportions within the beyond-plot slice)
bp_article = (
    bp_df.groupby(["article_id", "tag_short"])
    .size()
    .unstack(fill_value=0)
)
bp_article_pct = bp_article.div(bp_article.sum(axis=1), axis=0)

# Ordered: Prior Knowledge & Evaluative tags first, then Inference & Curiosity tags
bp_order = ["Background", "Personal", "Prescriptive", "Reactive", "Explanatory", "Predictive", "Curiosity"]
bp_colors = ["#f58518", "#ff9e4a", "#ffbf7f", "#ffd699",  # warm: Prior Knowledge & Evaluative
             "#e45756", "#ff7f7f", "#ffb0b0"]              # red: Inference & Curiosity

bp_long = bp_article_pct.reset_index().melt(
    id_vars="article_id", var_name="category", value_name="proportion"
)

alt.Chart(bp_long, title="Beyond-plot sub-category mix per article").mark_bar().encode(
    x=alt.X("article_id:N", sort=article_order, title="Article"),
    y=alt.Y("proportion:Q", stack="normalize", title="Share within beyond-plot"),
    color=alt.Color("category:N", sort=bp_order,
                    scale=alt.Scale(domain=bp_order, range=bp_colors),
                    title="Category"),
    tooltip=["article_id", "category", alt.Tooltip("proportion:Q", format=".1%")],
).properties(width=700, height=350)

── Overall beyond-plot sub-category counts ──


,count,group,pct
tag_short,,,
Curiosity,71608,Inference & Curiosity,27.7%
Background,52876,Prior Knowledge & Evaluative,20.5%
Personal,48955,Prior Knowledge & Evaluative,19.0%
Predictive,33797,Inference & Curiosity,13.1%
Explanatory,29026,Inference & Curiosity,11.2%
Reactive,13281,Prior Knowledge & Evaluative,5.1%
Prescriptive,8626,Prior Knowledge & Evaluative,3.3%


alt.Chart(...)

In [12]:
# ── Heatmap: beyond-plot sub-category proportion by article ─────────────
alt.Chart(bp_long, title="Beyond-plot sub-category proportions (heatmap)").mark_rect().encode(
    x=alt.X("article_id:N", sort=article_order, title="Article"),
    y=alt.Y("category:N", sort=bp_order, title="Sub-category"),
    color=alt.Color("proportion:Q", scale=alt.Scale(scheme="orangered"), title="Proportion"),
    tooltip=["article_id", "category", alt.Tooltip("proportion:Q", format=".1%")],
).properties(width=700, height=250)

alt.Chart(...)

In [13]:
# ── Cross-article consistency of beyond-plot sub-categories ─────────────
bp_cv = bp_article_pct[bp_order].describe().loc[["mean", "std"]].T
bp_cv["cv"] = bp_cv["std"] / bp_cv["mean"]
bp_cv["high_level"] = bp_cv.index.map(
    lambda t: "Prior Knowledge & Evaluative" if t in PRIOR_KNOWLEDGE_TAGS else "Inference & Curiosity"
)
bp_cv = bp_cv.sort_values("cv")
print("── Beyond-plot sub-category consistency (lower CV = more consistent) ──")
display(bp_cv.round(3))

# Also compare the two high-level beyond-plot groups
pk_frac = bp_article_pct[list(PRIOR_KNOWLEDGE_TAGS)].sum(axis=1)
hr_frac = bp_article_pct[list(HYPOTHETICAL_TAGS)].sum(axis=1)
print(f"\nPrior Knowledge & Evaluative share of beyond-plot: mean={pk_frac.mean():.1%}, std={pk_frac.std():.1%}")
print(f"Inference & Curiosity share of beyond-plot: mean={hr_frac.mean():.1%}, std={hr_frac.std():.1%}")

# Example sentences
print("\n── Sample sentences from each beyond-plot sub-category ──")
for cat in bp_order:
    full_tag = [k for k, v in SHORT.items() if v == cat][0]
    subset = bp_df[bp_df["tag_short"] == cat]
    samples = subset["original_comment"].sample(min(4, len(subset)), random_state=42)
    group_label = "Prior Knowledge & Evaluative" if cat in PRIOR_KNOWLEDGE_TAGS else "Inference & Curiosity"
    print(f"\n▸ {cat} ({full_tag}) [{group_label}]:")
    for s in samples:
        print(f"  • {s}")

KeyError: "['Prescriptive', 'Reactive'] not in index"

## RQ3 — Carving visual observations

*What kinds of visual observations do people make? How do the three VO sub-types distribute?*

- **VO1: Chart Structure & Text** — what the chart IS, how it's organized, what labels/legends say
- **VO2: Data Point Extraction** — extracting a specific numeric value or magnitude
- **VO3: Cross-point Pattern Recognition** — comparing data points, tracing trends, characterizing shape

In [ ]:
# ── Visual observation sub-type distribution ────────────────────────────
vo_df = df[df["group"] == "Visual Observation"].copy()

# Overall VO breakdown
vo_overall = vo_df["tag_short"].value_counts()
print("── Visual observation sub-type counts ──")
display(vo_overall.to_frame("count").assign(pct=lambda d: (d["count"] / d["count"].sum()).map("{:.1%}".format)))

# Per-article VO breakdown
vo_article = (
    vo_df.groupby(["article_id", "tag_short"])
    .size()
    .unstack(fill_value=0)
)
vo_article_pct = vo_article.div(vo_article.sum(axis=1), axis=0)

vo_long = vo_article_pct.reset_index().melt(
    id_vars="article_id", var_name="vo_type", value_name="proportion"
)

bars = alt.Chart(vo_long, title="VO sub-type mix per article").mark_bar().encode(
    x=alt.X("article_id:N", sort=article_order, title="Article"),
    y=alt.Y("proportion:Q", stack="normalize", title="Share within visual observations"),
    color=alt.Color(
        "vo_type:N",
        sort=["VO1", "VO2", "VO3"],
        scale=alt.Scale(
            domain=["VO1", "VO2", "VO3"],
            range=["#9ecae1", "#4292c6", "#08519c"],
        ),
        title="VO type",
    ),
    tooltip=["article_id", "vo_type", alt.Tooltip("proportion:Q", format=".1%")],
).properties(width=700, height=300)

bars

In [ ]:
# ── VO3 dominance: is trend/pattern observation always the largest VO? ──
vo_dominant = vo_article.idxmax(axis=1).value_counts()
print("── Which VO sub-type dominates per article? ──")
display(vo_dominant.to_frame("n_articles"))

# VO rate per sentence (how many of ALL sentences are VO?)
vo_rate = (
    df.groupby("article_id")["group"]
    .apply(lambda s: (s == "Visual Observation").mean())
    .reset_index()
)
vo_rate.columns = ["article_id", "vo_rate"]
print(f"\nVisual observation rate across articles:")
print(f"  Mean:   {vo_rate['vo_rate'].mean():.1%}")
print(f"  Median: {vo_rate['vo_rate'].median():.1%}")
print(f"  Range:  [{vo_rate['vo_rate'].min():.1%}, {vo_rate['vo_rate'].max():.1%}]")

In [ ]:
# ── Full 9-category heatmap across all articles ─────────────────────────
full_ct = (
    df.groupby(["article_id", "tag_short"])
    .size()
    .unstack(fill_value=0)
)
full_pct = full_ct.div(full_ct.sum(axis=1), axis=0)

tag_order = [
    "VO1", "VO2", "VO3",
    "Background", "Personal", "Prescriptive", "Reactive",
    "Explanatory", "Predictive", "Curiosity",
]

heat_df = full_pct.reset_index().melt(
    id_vars="article_id", var_name="category", value_name="proportion"
)
heat_df = heat_df[heat_df["category"].isin(tag_order)]

alt.Chart(heat_df, title="Category profile per article (normalised, excl. Meta & Uncategorizable)").mark_rect().encode(
    x=alt.X("article_id:N", sort=article_order, title="Article"),
    y=alt.Y("category:N", sort=tag_order, title="Category"),
    color=alt.Color("proportion:Q", scale=alt.Scale(scheme="viridis"), title="Proportion"),
    tooltip=["article_id", "category", alt.Tooltip("proportion:Q", format=".1%")],
).properties(width=700, height=300)

In [ ]:
# ── Sample VO sentences by sub-type (to illustrate the categories) ──────
print("── Sample sentences from each VO sub-type ──\n")
for vo_type in ["VO1", "VO2", "VO3"]:
    full_tag = [k for k, v in SHORT.items() if v == vo_type][0]
    subset = vo_df[vo_df["tag_short"] == vo_type]
    samples = subset.sample(min(8, len(subset)), random_state=42)
    print(f"▸ {vo_type} — {full_tag} ({len(subset):,} sentences)")
    for _, row in samples.iterrows():
        print(f"  [art {row['article_id']}] {row['original_comment']}")
    print()

## RQ4 — Within-category divergence (behavioural coding)

*Within the same category on the same plot, do commenters make diverging or contradictory statements?*

We approach this by:
1. **Extracting the most-discussed topics per article** using simple keyword clustering on VO3 sentences (the richest observation category).
2. **Surfacing candidate divergent pairs** — sentences in the same category on the same article that use opposing valence/direction words (e.g. "increase" vs "decrease", "more" vs "less", "higher" vs "lower").

In [ ]:
# ── Candidate divergent pairs within the same category + article ────────
import re

OPPOSING_PAIRS = [
    (r"\bincrease", r"\bdecrease"),
    (r"\brising", r"\bfalling"),
    (r"\bhigher", r"\blower"),
    (r"\bmore\b", r"\bless\b"),
    (r"\bgrew", r"\bshrank"),
    (r"\bgrowth", r"\bdecline"),
    (r"\bgain", r"\bloss"),
    (r"\bup\b", r"\bdown\b"),
    (r"\babove", r"\bbelow"),
    (r"\bpositive", r"\bnegative"),
]


def find_divergent_pairs(sub_df: pd.DataFrame, n_max: int = 5):
    """Find sentence pairs from different commenters using opposing direction words."""
    sentences = sub_df[["comment_id", "original_comment"]].values.tolist()
    pairs = []
    for i in range(len(sentences)):
        for j in range(i + 1, len(sentences)):
            if sentences[i][0] == sentences[j][0]:
                continue  # same commenter
            s1 = sentences[i][1].lower()
            s2 = sentences[j][1].lower()
            for pat_a, pat_b in OPPOSING_PAIRS:
                if (re.search(pat_a, s1) and re.search(pat_b, s2)) or \
                   (re.search(pat_b, s1) and re.search(pat_a, s2)):
                    pairs.append((sentences[i], sentences[j]))
                    break
            if len(pairs) >= n_max:
                return pairs
    return pairs


# Run on VO3 sentences (the category most likely to show genuine divergence)
print("── Candidate divergent VO3 pairs (different commenters, same article) ──\n")
divergent_results = []
for aid in sorted(df["article_id"].unique()):
    sub = vo_df[(vo_df["article_id"] == aid) & (vo_df["tag_short"] == "VO3")]
    if len(sub) < 4:
        continue
    pairs = find_divergent_pairs(sub, n_max=3)
    if pairs:
        divergent_results.append((aid, pairs))

for aid, pairs in divergent_results[:8]:
    print(f"Article {aid}:")
    for (cid1, s1), (cid2, s2) in pairs:
        print(f"  [c{cid1}] {s1}")
        print(f"  [c{cid2}] {s2}")
        print()

print(f"\nArticles with divergent VO3 pairs: {len(divergent_results)} / {df['article_id'].nunique()}")

In [ ]:
# ── Also check divergence in Explanatory inference ──────────────────────
expl_df = df[df["tag_short"] == "Explanatory"].copy()

print("── Candidate divergent Explanatory pairs (same article, diff commenters) ──\n")
expl_div = []
for aid in sorted(df["article_id"].unique()):
    sub = expl_df[expl_df["article_id"] == aid]
    if len(sub) < 4:
        continue
    pairs = find_divergent_pairs(sub, n_max=2)
    if pairs:
        expl_div.append((aid, pairs))

for aid, pairs in expl_div[:6]:
    print(f"Article {aid}:")
    for (cid1, s1), (cid2, s2) in pairs:
        print(f"  [c{cid1}] {s1}")
        print(f"  [c{cid2}] {s2}")
        print()

print(f"Articles with divergent explanatory pairs: {len(expl_div)} / {df['article_id'].nunique()}")

## Summary statistics

In [ ]:
# ── Category-transition analysis (within comments) ─────────────────────
transitions = Counter()
for (aid, cid), grp in df.groupby(["article_id", "comment_id"]):
    tags = grp["tag_short"].tolist()
    for a, b in zip(tags, tags[1:]):
        transitions[(a, b)] += 1

trans_df = pd.DataFrame(
    [(a, b, c) for (a, b), c in transitions.items()],
    columns=["from", "to", "count"],
)
trans_df = trans_df[trans_df["from"].isin(tag_order) & trans_df["to"].isin(tag_order)]

trans_total = trans_df.groupby("from")["count"].transform("sum")
trans_df["prob"] = trans_df["count"] / trans_total

heat_trans = trans_df.copy()
heat_trans["from"] = pd.Categorical(heat_trans["from"], categories=tag_order, ordered=True)
heat_trans["to"] = pd.Categorical(heat_trans["to"], categories=tag_order, ordered=True)

alt.Chart(heat_trans, title="Category transition probabilities (within comments)").mark_rect().encode(
    x=alt.X("to:N", sort=tag_order, title="Next category"),
    y=alt.Y("from:N", sort=tag_order, title="Current category"),
    color=alt.Color("prob:Q", scale=alt.Scale(scheme="oranges"), title="P(next | current)"),
    tooltip=["from", "to", alt.Tooltip("prob:Q", format=".2f"), "count:Q"],
).properties(width=450, height=350)

In [ ]:
# ── Summary table: per-article statistics ───────────────────────────────
summary_rows = []
for aid in sorted(df["article_id"].unique(), key=int):
    sub = df[df["article_id"] == aid]
    n_sentences = len(sub)
    n_commenters = sub["comment_id"].nunique()
    vo_frac = (sub["group"] == "Visual Observation").mean()
    pk_frac = (sub["group"] == "Prior Knowledge & Evaluative").mean()
    hr_frac = (sub["group"] == "Inference & Curiosity").mean()
    
    vo_sub = sub[sub["group"] == "Visual Observation"]["tag_short"]
    dom_vo = vo_sub.value_counts().index[0] if len(vo_sub) > 0 else "—"
    
    bp_sub = sub[sub["group"] != "Visual Observation"]["tag_short"]
    dom_bp = bp_sub.value_counts().index[0] if len(bp_sub) > 0 else "—"
    
    summary_rows.append({
        "article": aid,
        "sentences": n_sentences,
        "commenters": n_commenters,
        "visual_obs_%": f"{vo_frac:.0%}",
        "prior_knowledge_eval_%": f"{pk_frac:.0%}",
        "inference_curiosity_%": f"{hr_frac:.0%}",
        "dominant_VO": dom_vo,
        "dominant_beyond": dom_bp,
    })

summary_table = pd.DataFrame(summary_rows)
display(summary_table)